<img style="float: right;" src="img/python.png">

### NoSQL am Beispiel von Mongo-DB
Im Gegensatz zu SQL-Datenbanken wie SQLite, MySQL oder MariaDB handelt es sich bei MongoDB **NICHT um eine relationale Datenbank**.
MongoDB ist eine **dokumentorientierte Datenbank mit einer nicht strukturierten Abfragesprache**.

MongoDB ist eine beliebte NoSQL-Datenbank, die für ihre Flexibilität und Skalierbarkeit bekannt ist. In Kombination mit Python, einer der vielseitigsten Programmiersprachen, bietet sie eine leistungsstarke Plattform zur Verwaltung und Verarbeitung von Daten.

Da es MongoDB auf einen Datenbank-Server basiert, muss dieser installiert sein. Der MongoDB-Server kann unter der folgenden Adresse runtergeladen werden:

[Download MongoDB-Community Edition](https://www.mongodb.com/try/download/community)

**Glücklicherweise ist MongoDB bereits auf unserer virtuellen Maschine installiert**.

Nach der Installation kann der MongoDB Compass aufgerufen werden:
<br>
<img style="float: center;" src="img/mongodb0.png">
<br>

**Und wieder geschlossen, da wir ja in mit Python und PyCharm arbeiten**. ;-)



#### Verbindung zu Datenbank mit Hilfe von PyCharm

<br>
<img style="float: center;" src="img/mongodb6.png">
<br>
<img style="float: center;" src="img/mongodb7.png">
<br>



Bei der ersten Verbindung zur Datenbank muss noch der passende Treiber installiert und unter Umständen aktualisiert werden.

Zugangsdaten sind in diesen Fall nicht erforderlich, da der Server auf der virtuellen Maschine nicht geschützt ist.

### Wichtig zu beachten

MongoDB speichert Daten in einem Format namens **BSON**, das eine binäre Version von JSON ist.
MongoDB ist **schemafrei**, was bedeutet, dass Dokumente in derselben Collection unterschiedliche Felder haben können.
Die Interaktion mit MongoDB über Python erfolgt hauptsächlich durch das Arbeiten mit **Dictionaries** und **Cursors**.


#### Verbindung herstellen
In unserem einfachen Beispiel ist der Login zu Datenbank nicht über Zugangsdaten geschützt. Um auf die Datenbank
zuzugreifen nutzen wir **Mongo-Client** aus der Bibliothek "**pymongo**" die installiert werden muss.
Mehr Informationen zu pymongo finden Sie unter der folgenden Adresse:
[Pymongo-Readthedocs](https://pymongo.readthedocs.io/en/stable/tutorial.html)

In [ ]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["unsere_tool"]

#### Daten zur Datenbank hinzufügen

Mongodb unterstützt dabei folgende Datentypen:
<br>
<img style="float: center;" src="img/mongodb14.png">
Quelle: Entwickler.press

#### String Kodierung
Strings können beim Speichern UTF-8 kodiert sein. Mongodb verarbeitet diese automatisch. Strings müssen also nicht vorher in ASCII-Format konvertiert werden.

#### Datensatz einer Collection hinzufügen

Sobald Sie Daten in eine Collection einfügen, wird die Datenbank (in diesem Fall **neueDatenbank**) erstellt.

In [ ]:
collection = db["benutzer"]

collection.insert_one({"name": "Max Mustermann", "alter": 28, "rolle": "Administrator"})

**Bingo!**
<br>
<img style="float: center;" src="img/neue_Datenbank.PNG">
<br>
<img style="float: center;" src="img/mongodb8.png">

#### Embedded Documents
Es ist auch möglich Teildokumente in Top-Level-Dokumente einzufügen. Man könnte diese auch unendlich untereinander verschachteln.
In der Regel würde das aber den Aufwand, um an die Daten zu kommen erhöhen.

#### Erfolgskontrolle, alle Datenbanken auflisten:

In [ ]:
print(client.list_database_names())

Dieser Code zeigt alle Datenbanknamen in Ihrer MongoDB-Instanz an. Sobald die erste Dateneinfügung erfolgt ist, sollte neueDatenbank in dieser Liste erscheinen.

Denken Sie daran, dass in MongoDB eine Datenbank oder eine Collection **nicht physisch erstellt wird, bis Daten darin gespeichert werden**. Dies unterscheidet MongoDB von traditionellen relationalen Datenbanksystemen.

#### Mehrere Elemente der Datenbank hinzufügen

In [ ]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["unsere_tool"]

collection = db["benutzer"]

addlist =[
    {
        "name": "Tony Tester",
        "alter": 30,
        "rolle": "Benutzy"
        },

    {
        "name": "Tanja ´Tester",
        "alter": 25,
        "rolle": "Benutzy"
        }
]

collection.insert_many(addlist)

Wie Ihnen sicher aufgefallen ist, brauchen Tabellen, anders als bei SQL-Datenbanken, nicht erweitert werden, um einen Datensatz hinzuzufügen, der weitere Angaben enthält.
Hier wurde ein neuer Datensatz angelegt, der im Gegensatz zu den anderen Datensätzen noch eine Rolle enthält:

<img style="float: center;" src="img/mongodb9.png">

#### Daten aus der Datenbank lesen

### find_one()

In [ ]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["unsere_tool"]

collection = db["benutzer"]

daten = collection.find_one({"name": "Tony Tester"})
print(daten)

Wie Sie sicher  bemerkt haben, erfolgt die Ausgabe als **Dictionary**.

In [ ]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["unsere_tool"]

collection = db["benutzer"]

daten = collection.find_one({"name": "Tony Tester"})

for key, value in daten.items():
    print(f"{key}: {value}")


print(daten["name"])
print(daten["alter"])

#### Die Objectid

<img style="float: center;" src="img/mongodb15.png">
<br>
<br>
Die Objectid wird vom Client Driver oder der Datenbank automatisch generiert.
Falls gewünscht kann der Entwickler eine eigene Objectid generieren. Diese darf allerdings nur einmal vorkommen:


#### Zerlegen der ObjectID und Extrahieren des Timestamp

Der Zeitstempel wird in UTC ausgegeben. Das muss bei der Suche berücksichtigt werrden.

In [ ]:
from pymongo import MongoClient
from bson import ObjectId

client = MongoClient("mongodb://localhost:27017/")
db = client["unsere_tool"]

collection = db["benutzer"]

daten = collection.find_one({"name": "Tony Tester"})

obj_id = ObjectId(daten["_id"])
print(obj_id)

print(obj_id.generation_time)

#### Suchen nach Erstellungsdatum

In [ ]:
from pymongo import MongoClient
from bson import ObjectId
from datetime import datetime

client = MongoClient("mongodb://localhost:27017/")
db = client["unsere_tool"]

collection = db["benutzer"]

daten = collection.find_one({"name": "Tony Tester"})

obj_id = ObjectId(daten["_id"])

# Startdatum:
start_date = datetime(2026, 1,26)

# umschlüsseln
start_object = ObjectId.from_datetime(start_date)

documents = collection.find({"_id": {"$gte": start_object}})

for doc in documents:
    print(doc)

#### ... und Zeit

In [ ]:
from pymongo import MongoClient
from bson import ObjectId
from datetime import datetime

client = MongoClient("mongodb://localhost:27017/")
db = client["unsere_tool"]

collection = db["benutzer"]

obj_id = ObjectId(daten["_id"])

# Startdatum und Zeit:
start_date = datetime(2026, 1,26, 8, 0, 0)

# umschlüsseln
start_object = ObjectId.from_datetime(start_date)

documents = collection.find({"_id": {"$gte": start_object}})

for doc in documents:
    print(doc)

Ändern der Zeitangabe in UTC.

In [ ]:
from pymongo import MongoClient
from bson import ObjectId
from datetime import datetime
import pytz


client = MongoClient("mongodb://localhost:27017/")
db = client["unsere_tool"]

collection = db["benutzer"]

berlin = pytz.timezone('Europe/Berlin')  # Deutsche Zeitzone einstellen


# Startdatum und Zeit:
start_date = berlin.localize(datetime(2026, 1,27, 8, 0, 0))
end_date = berlin.localize(datetime(2026, 1,27, 18, 0, 0))


# Umgewandelte Zeitstempel für die Suche in UTC
start_utc = start_date.astimezone(pytz.utc)
end_utc = end_date.astimezone(pytz.utc)


daten = collection.find({"_id": {
    "$gte": ObjectId.from_datetime(start_utc),
    "$lte": ObjectId.from_datetime(end_utc)
}})

for doc in daten:
    print(doc)

#### Mehrere Suchparameter
Natürlich können auch mehrere Suchparameter kombiniert werden:

In [ ]:
from pymongo import MongoClient
# from bson import ObjectId
# from datetime import datetime
# import pytz


client = MongoClient("mongodb://localhost:27017/")
db = client["unsere_tool"]

collection = db["benutzer"]

daten = collection.find({"name": "Tanja Tester", "alter": 25})


for i in daten:
    for key, value in i.items():
        print(f"{key}: {value}")

### Nach Werten filtern:

In [6]:
import pymongo
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client['unsere_tool']

collection = db['benutzer']

daten = collection.find({"name":"Max Power"})

print("daten", daten) # Ausgabe Suchobjekt

print("daten 0", daten[0]) #Ausgabe erster Listeneintrag -> Objekt

for doc in daten: # Ausgabe aller gefundenen Einträge
    print("doc", doc)


{'_id': ObjectId('69773d1f4e63f5a649cf846e'), 'name': 'Max Power', 'alter': 30, 'rolle': 'Administrator'}
{'_id': ObjectId('69773d1f4e63f5a649cf846e'), 'name': 'Max Power', 'alter': 30, 'rolle': 'Administrator'}


In [ ]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["unsere_tool"]

collection = db["benutzer"]


daten = collection.find({"alter": {"$gte": 30}})


for i in daten:
    for key, value in i.items():
        print(f"{key}: {value}")

In [ ]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client['unsere_tool']

collection = db['benutzer']

daten = collection.find({"alter": {"$gte": 30}})

print(daten)  # Ausgabe Suchobjekt

print(daten[0])  # Ausgabe erster Listeneintrag -> Objekt

for doc in daten:  # Ausgabe aller gefundenen Einträge
    print(doc)

### Weitere Filter- und Vergleichsmöglichkeiten:

In [ ]:
collection.find({"alter": {"$eq": 25}})  # gleich
collection.find({"alter": {"$ne": 25}})  # nicht gleich
collection.find({"alter": {"$gt": 25}})  # größer als
collection.find({"alter": {"$gte": 25}})  # größer oder gleich
collection.find({"alter": {"$lt": 25}})  # kleiner als
collection.find({"alter": {"$lte": 25}})  # kleiner oder gleich
collection.find({"alter": {"$in": [20, 25, 30]}})  # in einer Liste enthalten
collection.find({"alter": {"$nin": [20, 25, 30]}})  # nicht in einer Liste enthalten

#### Alle Daten der Datenbank anzeigen mit find():

In [4]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client['unsere_tool']

collection = db['benutzer']

daten = collection.find()

for i in daten:
    for key, value in i.items():
        print(f"{key}: {value}")


_id: 69773d1f4e63f5a649cf846e
name: Max Power
alter: 30
rolle: Administrator
_id: 6977456d682a4fad32a529a3
name: Pablo Picasso
alter: 39
rolle: tester
_id: 6977456d682a4fad32a529a4
name: Pedro Picapiedra
alter: 69
rolle: Benutzer
_id: 697745d46bcd0c93c2d179bc
name: Juan Juarez
alter: 68
_id: 6978776cf860d04c1d5a385b
name: Pedro el Escamoso
alter: 19
rolle: tanzer


Der Syntax von find():
find(query, fields, num-docs, skip-docs)

#### Anzahl der Dokumente in einer Datenbank

In [2]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client['unsere_tool']

collection = db['benutzer']

print(collection.count_documents({}))

5


#### Ein kleines Skript zum Hinzufügen neuer Benutzer (Übung)

In [5]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["unsere_tool"]

collection = db["benutzer"]

nombre = input("escribe el nombre completo")
edad = int(input("escribe la edad"))
puesto = input("escribe el puesto")

nutzer_info = {"name": nombre, "alter": edad, "rolle": puesto}

collection.insert_one(nutzer_info)


InsertOneResult(ObjectId('6978a24f1dc7db8915a9abce'), acknowledged=True)

#### Ein kleines Skript zum Suchen eines Eintrages (gemeinsame Übung)

In [13]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["unsere_tool"]

collection = db["benutzer"]

nombre = input("escribe el nombre completo a buscar")

# busca si "nombre" esta de alguna forma dentro del str.
daten = collection.find({"name": {"$regex": f".*{nombre}.*"}})
print(daten)

for i in daten:
    for k, v in i.items():
        print(f"{k}: {v}")


_id: 6977456d682a4fad32a529a4
name: Pedro Picapiedra
alter: 69
rolle: Benutzer
_id: 6978776cf860d04c1d5a385b
name: Pedro el Escamoso
alter: 19
rolle: tanzer


.* bedeutet: beliebige Zeichenfolge beliebiger Länge <br>
(. = ein beliebiges Zeichen, * = beliebig oft).

#### Wie viele Python-DozenInnen befinden sich in der Datenbank?

In [ ]:
print(collection.count_documents({"rolle": "Python-DozentIn"}))

### Daten ändern
Jede Datenbank muss nicht nur aufgerufen, sondern auch gepflegt werden (OK, vielleicht nicht alle, aber die meisten).
Zum Ändern von Daten wird die **update()**-Methode verwendet.
Naturgemäß erwartet die Methode **2**, oder **maximal 4** Parameter.

Syntax:
update(query, new-document/update-document, upsert, multiple)
<br>

<img style="float: center;" src="img/mongodb19.png">




In [15]:
# Upgrade erste Juan Juarez zum Administrator
"""collection.update_one({"name": "Juan Juarez"}, {"$set": {"rolle": "Administrator"}})"""

# Upgrade alle Pedro Picapiedra zum Administrator
"""collection.update_many({"name": "Pedro Picapiedra"}, {"$set": {"rolle": "Administrator"}})"""

# Upgrade Pedro Picapiedra alter 69 zum Administrator
"""collection.update_one({"name": "Pedro Picapiedra", "alter": 69},{"$set": {"rolle": "Administrator"}})"""


UpdateResult({'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

### Daten löschen
#### Einen Eintrag löschen

Wir löschen den Eintrag von Tony Tester:

In [4]:
# para borrar el primero
collection.delete_one({"alt": "33"})

DeleteResult({'n': 1, 'ok': 1.0}, acknowledged=True)

#### Mehrere Einträge löschen

Wir löschen alle Python-DozentInnen:

In [ ]:
# para borrar varios
"""collection.delete_many({"rolle": "Python-DozentInnen"})"""

Natürlich funktionieren auch Filter:

In [2]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client['unsere_tool']

collection = db['benutzer']

# Alle Administratoren löschen
collection.delete_many({"alt": {"$gte": 33}})

DeleteResult({'n': 0, 'ok': 1.0}, acknowledged=True)

#### Indizierung:
Wie bei relationalen Datenbanken ist die Indizierung in MongoDB entscheidend für die Leistungsfähigkeit bei Abfragen. Sie können Indizes auf Collections erstellen, um die Abfragegeschwindigkeit zu verbessern.

In [ ]:
collection.create_index([("name", pymongo.ASCENDING)])

Jede Collection in MongoDB kann mehrere Indizes haben, die die Abfrageleistung verbessern. Sie können die Methode **index_information()** verwenden, um Details zu den Indizes einer bestimmten Collection zu erhalten.

Hier ist ein Beispiel, wie Sie die Informationen zu den Indizes einer Collection in MongoDB mit Python und pymongo abrufen können:

In [ ]:
indexes = collection.index_information()

# Informationen zu jedem Index ausgeben
for index in indexes:
    print(indexes[index])

- **index_information()** gibt ein Dictionary zurück, in dem die Schlüssel die Namen der Indizes sind und die Werte Details zu jedem Index enthalten.
- Die Ausgabe enthält Informationen wie den Namen des Indexes, die Felder, die indiziert sind, und ob der Index in aufsteigender (1) oder absteigender (-1) Reihenfolge ist.
- Standardmäßig hat jede Collection in MongoDB einen Index auf dem _id-Feld.

Diese Methode ist nützlich, um zu verstehen, wie Ihre Daten in MongoDB organisiert und indiziert sind, was wiederum Einfluss auf die Leistung Ihrer Abfragen hat.

### Wichtig!
Das sind die Grundlagen, um mit MongoDB in Python zu arbeiten. Es gibt noch viele weitere Funktionen und Methoden, die MongoDB bietet, insbesondere im Hinblick auf Aggregationen, Indizierung, und Verwaltung großer Datenmengen.

<img style="float: center;" src="img/wbs-logo.jpg">


### Abbildungs- und Quellenverzeichnis
https://de.wikipedia.org/wiki/Python_(Programmiersprache)
Das Python Logo ist ein eingetragenes Warenzeichen der Python Software Foundation
Alle auf dieser Website veröffentlichten Logos sowie Marken-, Produkt- und Warenzeichen sind Eigentum der jeweiligen Unternehmen
© WBS TRAINING AG – Alle Rechte vorbehalten

### Nutzungsrechte:
Die Nutzung dieser Dokumentation ist ausschließlich für Schulungszwecke der WBS TRAINING AG gestattet. Eine Weitergabe an Dritte, auch auszugsweise, sowie Vervielfältigungen und Verbreitungen aller Art (elektronische und andere Verfahren) inklusive Übersetzungen sind nur mit vorheriger schriftlicher Zustimmung des Rechtinhabers gestattet. Zuwiderhandlungen verpflichten zu Schadenersatz.

### Herausgeber:

WBS TRAINING AG
Lorenzweg 5
12099 Berlin
Haftungsausschluss:
Alle Inhalte sind nach bestem Wissen korrekt und vollständig recherchiert und mit größtmöglicher Sorgfalt für die Schulungsunterlage zusammengestellt. Wir sind um die laufende Aktualisierung aller Informationen und Daten bemüht. Dennoch können Fehler (z.B. Abweichungen zur beschriebenen Hard- und Software durch kurzfristige Updates) auftreten, sodass wir für die vollständige Übereinstimmung, Richtigkeit und Aktualität keine Gewähr übernehmen. Hinweise unserer Nutzer werden konsequent weiterverfolgt.
